In [ ]:

from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import seaborn as sns

from project_paths import paths 

df = pl.read_parquet(paths.processed_data / "unified_aims_eir_bgs.parquet")


df.shape


In [ ]:



def grade_expr(col: str) -> pl.Expr:
    return (
        pl.col(col)
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.extract(r"^(\d+)")
        .cast(pl.Int8, strict=False)
    )


In [ ]:

GEOSURE = [
    "geosure_collapsible_deposits__class",
    "geosure_compressible_ground__class",
    "geosure_landslides__class",
    "geosure_running_sand__class",
    "geosure_shrink_swell__class",
    "geosure_soluble_rocks__class",
]

df = df.with_columns(
    grade_expr("aims__current_condition").alias("grade_aims"),
    grade_expr("eir__condition_grade").alias("grade_eir"),
    *[pl.col(c).cast(pl.Int8, strict=False).alias(c) for c in GEOSURE],
)

df.select(["grade_aims", "grade_eir"])


In [ ]:


check = df.filter(pl.col("grade_eir").is_in([888, 999])).height
assert check == 0, f"{check} EIR redactions in final data somehow"


In [ ]:

# Grade 0 means "unchecked" (documented source value, not an error)#
# measure occurences, then exclude from distribution analysis 
# keep df intact so all rows can be used later 

df.group_by("grade_aims").len().sort("grade_aims")



graded = df.filter(pl.col("grade_aims").is_between(1, 5))
graded.height

In [ ]:

import geopandas as gpd

aims_geom = gpd.read_file(paths.aims_data / "aims.gpkg")[["asset_id", "geometry"]]
aims_geom["asset_id"] = aims_geom["asset_id"].astype("int64")




grade_df = (
    graded.select(
        pl.col("aims__asset_id").alias("asset_id"),
        "grade_aims", 
        "aims__asset_sub_type"
    )
    .to_pandas()
)

print(aims_geom, grade_df)


In [ ]:

export = aims_geom.merge(grade_df, on="asset_id", how="inner")
export.crs  


In [ ]:

export.to_file(paths.processed_data / "graded_assets.gpkg", driver="GPKG", layer="graded")

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R

library(sf); library(ggplot2); library(ggspatial); library(dplyr)

assets  <- st_read("data/processed/graded_assets.gpkg", layer = "graded")
camels  <- st_read("data/external/camels_gb_catchments.gpkg")  
camels  <- st_transform(camels, st_crs(assets))                 

assets$grade_aims <- factor(assets$grade_aims, levels = 1:5)

ggplot() +
    geom_sf(data = camels, fill = NA, linewidth = 0.1, colour = "grey70") +  
    geom_sf(data = assets, aes(colour = grade_aims), linewidth = 0.3) +
    scale_colour_viridis_d(option = "C", direction = -1) +                   
    coord_sf(datum = st_crs(27700)) +
    theme_void()